## Question

Each time you flip a coin, you get $1. If you get 3 heads in a row, the game ends and you lose everything. Under optimal play, what are your expected winnings?

## Solution
    x = current winnings   
    k = number of consecutive heads so far; k is 0, 1, or 2

V(x, k) is the maximum expected final winnings from this state.
At each state, choose the better of stopping (keep x) and flipping again:

    V(x, k) = max(x, C(x, k))

where C(x, k) is the expected value of continuing:

    C(x, 0) = 0.5 * V(x + 1, 0) + 0.5 * V(x + 1, 1)
    C(x, 1) = 0.5 * V(x + 1, 0) + 0.5 * V(x + 1, 2)
    C(x, 2) = 0.5 * V(x + 1, 0) + 0.5 * 0

First term in each equation is a tail: winnings increase by $1 and
the consecutive-head count resets to 0. A head increases the streak,
except when k = 2, where it makes 3 heads in a row and winnings = 0.

Choose a sufficiently large stopping limit N:
    V(N, k) = N for k = 0, 1, 2
Then calculate backward for x = N - 1 down to 0, using the equations.
V(0, 0) is the estimated optimal expected winnings at the start.
Try increasing N and check that V(0, 0) and the stop/continue decisions
remain stable.

### Optimal Play:

Always continue when:   
k = 0,1   
k = 2, x < 7

Stop when:
k = 2, x >= 7

### Expected Winnings: $7.2422

## Python dynamic-programming calculation


In [3]:
def solve_game(N):
    V = [[0.0 for _ in range(3)] for _ in range(N + 1)]

    for k in range(3):
        V[N][k] = float(N)

    for x in range(N - 1, -1, -1):
        continue_k0 = 0.5 * V[x + 1][0] + 0.5 * V[x + 1][1]
        continue_k1 = 0.5 * V[x + 1][0] + 0.5 * V[x + 1][2]
        continue_k2 = 0.5 * V[x + 1][0] + 0.5 * 0  # lose it all with heads

        V[x][0] = max(float(x), continue_k0)
        V[x][1] = max(float(x), continue_k1)
        V[x][2] = max(float(x), continue_k2)

    return V

for N in (10, 20, 50, 100, 200):
    print(f"N={N:3d} -> Expected winnings: ${solve_game(N)[0][0]:.6f}")

N= 10 -> Expected winnings: $5.912109
N= 20 -> Expected winnings: $7.062981
N= 50 -> Expected winnings: $7.241877
N=100 -> Expected winnings: $7.242187
N=200 -> Expected winnings: $7.242187


## Python Monte Carlo simulation

In [2]:
import random

def play_once():
    x = 0
    k = 0

    while True:
        if k == 2 and x >= 7:
            return x

        is_head = random.random() < 0.5
        x += 1

        if not is_head:
            k = 0
        elif k == 2:
            return 0
        else:
            k += 1

number_of_games = 200_000
average_payoff = sum(play_once() for _ in range(number_of_games)) / number_of_games
print(f"Average over {number_of_games:,} simulated games: ${average_payoff:.4f}")

Average over 200,000 simulated games: $7.2433


**Interpretation:** Compared to Expected winnings of about $7.2422